# Load Packages and Data

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root, so src/ and analysis/ import

<module 'src.pitch_suggestions' from '/Users/kids/Pitcher Similarity/src/pitch_suggestions.py'>

In [2]:
from src.data import build_all
from src.pitch_suggestions import suggest_pitches, plot_pitch_clusters
from src.pitch_suggestions import _find_target, _find_biomech_comps, _collect_pitches, _tag_novelty, _cluster_novel
import warnings
import pandas as pd

from src.distances import compute_euclidean_distances

pd.set_option('display.max_columns', None)

warnings.filterwarnings("ignore")

In [3]:
data = build_all(live=False)

statcast_clean  = data['statcast_clean']
pitch_type_summ = data['pitch_type_summ']
pitcher_summ    = data['pitcher_summ']
pitcher_summ_r  = data['pitcher_summ_r']
pitcher_summ_l  = data['pitcher_summ_l']
pitch_type_r  = data['pitch_type_r']
pitch_type_l  = data['pitch_type_l']

statcast_clean_25 = statcast_clean[statcast_clean['game_year'] == 2025]

def pid(name, pool):
    """Resolve a player name to its unique pitcher id within a pitcher_summ pool."""
    ids = pool.loc[pool['player_name'] == name, 'pitcher'].unique()
    if len(ids) != 1:
        raise ValueError(f"{name!r} maps to {len(ids)} pitcher ids: {sorted(ids)}")
    return int(ids[0])


# Identify Pitch Opportunities

In [4]:
# Canonical feature lists live in pitch_suggestions; import rather than redefine
# so this notebook can't drift from the app.
from src.pitch_suggestions import BIOMECH_FEATURES, PITCH_CHAR_FEATURES, PARAMS

### Bello Example

Example of what the tool recommends

In [5]:
b_bello = suggest_pitches(
    target_pitcher_id=pid('Bello, Brayan', pitcher_summ_r),
    pitcher_summ=pitcher_summ_r,
    pitch_type_summ=pitch_type_r,
    **PARAMS,
    mask=True
)

In [6]:
plot_pitch_clusters(b_bello)

In [7]:
b_bello['status']

'ok'

In [8]:
b_bello['target_info']

pitcher                        678394
p_throws                            R
player_name             Bello, Brayan
game_year                        2026
release_pos_x               -1.693049
release_pos_z                5.640953
release_extension            6.206897
arm_angle                   40.191786
n                                1102
max_velo                    94.857734
max_spin                  2438.546875
pri_fb                             SI
fb_pfx_x                    -1.364161
fb_n                            459.0
pri_fb_cd                           0
active_spin_fastball             82.3
FB_type                            FF
Name: 3483, dtype: object

In [9]:
b_bello['comps'].sort_values('comp_pitcher')[20:25]

,comp_pitcher,comp_year,distance
170,519151,2022,1.084322
44,519326,2021,0.672454
145,541640,2022,1.024090
243,542585,2023,1.247447
127,542888,2026,0.982544


In [10]:
b_bello['target_pitches']

,pitcher,player_name,game_year,pitch_type,release_speed,pfx_x,pfx_z,n
0,678394,"Bello, Brayan",2026,CH,88.562814,-1.354573,0.462111,199
1,678394,"Bello, Brayan",2026,CU,83.440625,0.148281,-0.691406,64
2,678394,"Bello, Brayan",2026,FC,87.174269,-0.012222,0.202515,171
3,678394,"Bello, Brayan",2026,FF,94.674627,-0.554925,0.933731,67
4,678394,"Bello, Brayan",2026,SI,94.857734,-1.364161,0.510240,459
5,678394,"Bello, Brayan",2026,ST,85.711268,0.647113,-0.316690,142


In [11]:
print(b_bello['novel_comp_pitches'].shape)
b_bello['novel_comp_pitches'].head()

(109, 20)


,pitcher,player_name,game_year,pitch_type,release_speed,pfx_x,pfx_z,n,total_n,usage_pct,min_dist_to_target,closest_target_pitch,cluster,_dist_to_centroid,_cluster_median_dist,_cluster_mad,_outlier_threshold,cluster_label,biomech_distance,sim_weight
0,429722,"Santana, Ervin",2021,CH,87.180000,-0.666571,1.075429,35,1063,0.032926,1.218081,CH,2,0.831853,0.470205,0.193488,1.2,CH,0.858524,1.164789
1,641585,"France, J.P.",2024,CH,82.346226,-1.249811,0.944811,106,466,0.227468,1.259385,CH,2,0.354170,0.470205,0.193488,1.2,CH,1.199381,0.833762
2,643615,"Zeuch, T.J.",2021,CH,81.695238,-1.075714,0.760952,21,286,0.073427,1.274129,CH,2,0.305342,0.470205,0.193488,1.2,CH,0.906285,1.103405
3,647336,"Soroka, Michael",2023,CH,82.030612,-1.304184,1.149694,98,537,0.182495,1.476957,CH,2,0.586241,0.470205,0.193488,1.2,CH,0.819229,1.220658
4,663752,"Morris, Cody",2022,CH,81.900000,-1.023333,0.830145,69,404,0.170792,1.297838,CH,2,0.225044,0.470205,0.193488,1.2,CH,0.566518,1.765166


In [12]:
b_bello['suggestions']

,cluster_label,cluster,n_comps,avg_release_speed,avg_pfx_x,avg_pfx_z,avg_min_dist_to_target,wavg_release_speed,wavg_pfx_x,wavg_pfx_z,pitch_types_in_cluster,comp_pitchers
0,Curveball,0,63,78.3,0.98,-1.09,1.53,78.6,0.99,-1.09,"Slow Curve, Curveball, Knuckle Curve, Slider","Anderson, Drew, Assad, Javier, Avila, Pedro, B..."
1,Sweeper,1,38,80.5,1.33,0.05,1.39,80.5,1.33,0.05,"Curveball, Slider, Sweeper, Slurve","Adam, Jason, Almonte, Yency, Bard, Luke, Barra..."
2,Changeup,2,8,83.2,-0.99,0.86,1.30,83.1,-0.99,0.84,"Changeup, Other","Bibee, Tanner, France, J.P., Morris, Cody, Rib..."


In [13]:
plot_pitch_clusters(b_bello)

In [14]:
b_bello['novel_comp_pitches'].sort_values('_dist_to_centroid', ascending=False)

,pitcher,player_name,game_year,pitch_type,release_speed,pfx_x,pfx_z,n,total_n,usage_pct,min_dist_to_target,closest_target_pitch,cluster,_dist_to_centroid,_cluster_median_dist,_cluster_mad,_outlier_threshold,cluster_label,biomech_distance,sim_weight
65,673540,"Senga, Kodai",2025,FO,82.506296,-0.830167,0.056204,540,1879,0.287387,1.259410,FC,2,1.175298,0.470205,0.193488,1.2,CH,0.790959,1.264287
43,667297,"Nance, Tommy",2025,CU,84.960741,0.991111,-1.236074,135,461,0.292842,1.288290,CU,0,1.145317,0.527126,0.162674,1.2,CU,0.365297,2.737489
19,605135,"Bassitt, Chris",2022,CU,71.751630,1.243397,-1.142120,368,2853,0.128987,2.436750,CU,0,1.138801,0.527126,0.162674,1.2,CU,0.739874,1.351580
11,506433,"Darvish, Yu",2021,CU,72.236607,0.841429,-1.252321,112,2749,0.040742,2.204076,CU,0,1.052281,0.527126,0.162674,1.2,CU,0.737974,1.355059
32,641585,"France, J.P.",2024,CU,74.211111,0.780111,-1.610889,90,466,0.193133,2.167572,CU,0,1.039053,0.527126,0.162674,1.2,CU,1.199381,0.833762
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21,605242,"Fulmer, Michael",2022,CU,78.868182,1.077273,-1.003182,22,1043,0.021093,1.409208,CU,0,0.195321,0.527126,0.162674,1.2,CU,0.784033,1.275455
79,572193,"Tepera, Ryan",2023,ST,80.315625,1.234375,0.130625,32,229,0.139738,1.308287,ST,1,0.164982,0.418092,0.151934,1.2,ST,1.211373,0.825509
56,681347,"Burrows, Mike",2026,CU,79.037611,0.902301,-1.105531,226,1567,0.144225,1.299197,CU,0,0.162587,0.527126,0.162674,1.2,CU,1.281736,0.780191
83,621107,"Eflin, Zach",2023,ST,79.710377,1.386509,0.045189,106,2577,0.041133,1.429582,ST,1,0.143840,0.418092,0.151934,1.2,ST,1.442200,0.693384


### Step Guide

In [15]:
target_row, target_year = _find_target(pitcher_summ_l, pid('Beeks, Jalen', pitcher_summ_l))

In [16]:
target_dists = _find_biomech_comps(
    pitcher_summ_l, pid('Beeks, Jalen', pitcher_summ_l), target_year,
    BIOMECH_FEATURES, biomech_distance_threshold=PARAMS['biomech_distance_threshold'],
    min_pitches=PARAMS['min_pitches'],
)

In [17]:
target_pitches, comp_pitches = _collect_pitches(
    pitch_type_l, target_pitcher_id=pid('Beeks, Jalen', pitcher_summ_l), target_year=target_year, target_dists=target_dists,
    pitch_features=PITCH_CHAR_FEATURES, min_comp_usage_pct=PARAMS['min_comp_usage_pct'], min_pitches=PARAMS['min_pitches']
)

In [18]:
from sklearn.preprocessing import StandardScaler
global_scaler = StandardScaler().fit(
    pitch_type_l[PITCH_CHAR_FEATURES].dropna().values
)

In [19]:
comp_pitches, novel = _tag_novelty(
    target_pitches, comp_pitches, PITCH_CHAR_FEATURES, novelty_distance_threshold=PARAMS['novelty_distance_threshold'], global_scaler=global_scaler
)

In [20]:
print(novel.shape)
novel.head()

(154, 12)


,pitcher,player_name,game_year,pitch_type,release_speed,pfx_x,pfx_z,n,total_n,usage_pct,min_dist_to_target,closest_target_pitch
4,501625,"Alvarez, Jose",2021,CH,81.165441,1.201103,0.359853,272,970,0.280412,1.318171,CH
9,543594,"Nolin, Sean",2021,CH,79.933333,1.294487,0.687436,78,496,0.157258,1.363042,CH
11,570257,"Rodríguez, Joely",2021,CH,88.355088,0.935123,0.026702,285,761,0.374507,1.217063,CH
12,571510,"Boyd, Matthew",2021,CH,79.934266,1.352063,0.499476,286,1272,0.224843,1.422488,CH
27,607644,"Means, John",2023,CH,80.900000,0.969735,1.375044,113,339,0.333333,1.516602,CH


In [21]:
novel = _cluster_novel(novel, global_scaler, PITCH_CHAR_FEATURES)

In [22]:
novel

,pitcher,player_name,game_year,pitch_type,release_speed,pfx_x,pfx_z,n,total_n,usage_pct,min_dist_to_target,closest_target_pitch,cluster,_dist_to_centroid,_cluster_median_dist,_cluster_mad,_outlier_threshold,cluster_label
0,477132,"Kershaw, Clayton",2022,CU,73.209333,-0.408833,-1.291967,300,1841,0.162955,2.283957,FC,1,0.913392,0.583124,0.174771,1.2,CU
1,518633,"Duffy, Danny",2021,CU,77.323853,-0.618532,-1.326147,109,1009,0.108028,2.037884,FC,1,0.494417,0.583124,0.174771,1.2,CU
2,519293,"Smith, Will",2021,CU,77.637815,-0.727563,-0.631681,119,1079,0.110287,1.221701,FC,1,0.553385,0.583124,0.174771,1.2,CU
3,543594,"Nolin, Sean",2021,CU,74.070149,-0.823582,-0.992239,67,496,0.135081,1.984186,FC,1,0.565868,0.583124,0.174771,1.2,CU
4,571510,"Boyd, Matthew",2021,CU,73.503125,-1.042500,-1.319896,96,1272,0.075472,2.485619,FC,1,0.870842,0.583124,0.174771,1.2,CU
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,800048,"Messick, Parker",2025,SL,86.349451,-0.580330,0.426044,91,625,0.145600,1.210762,FC,0,0.533861,0.334564,0.105373,1.2,FC
112,805326,"Schweitzer, Tyler",2026,SL,87.880952,-0.323810,0.392222,63,149,0.422819,1.310872,FC,0,0.258029,0.334564,0.105373,1.2,FC
113,813349,"Early, Connelly",2026,SL,87.097561,-0.024146,0.509024,123,1528,0.080497,1.306742,FC,0,0.297810,0.334564,0.105373,1.2,FC
114,663687,"Harris, Hogan",2023,ST,77.025641,-0.882393,-0.811368,117,1109,0.105500,1.551702,FC,1,0.330669,0.583124,0.174771,1.2,CU


## Testing

In [23]:
s_peralta = suggest_pitches(
    target_pitcher_id=pid('Peralta, Sammy', pitcher_summ_l),
    pitcher_summ=pitcher_summ_l,
    pitch_type_summ=pitch_type_l,
    **PARAMS,
    mask=True
)

In [24]:
e_lauer = suggest_pitches(
    target_pitcher_id=pid('Lauer, Eric', pitcher_summ_l),
    pitcher_summ=pitcher_summ_l,
    pitch_type_summ=pitch_type_l,
    **PARAMS,
    mask=True
)

In [25]:
r_hill = suggest_pitches(
    target_pitcher_id=pid('Hill, Rich', pitcher_summ_l),
    pitcher_summ=pitcher_summ_l,
    pitch_type_summ=pitch_type_l,
    **PARAMS,
    mask=True
)

In [26]:
p_corbin = suggest_pitches(
    target_pitcher_id=pid('Corbin, Patrick', pitcher_summ_l),
    pitcher_summ=pitcher_summ_l,
    pitch_type_summ=pitch_type_l,
    **PARAMS,
    mask=True
)

In [27]:
t_alexander = suggest_pitches(
    target_pitcher_id=pid('Alexander, Tyler', pitcher_summ_l),
    pitcher_summ=pitcher_summ_l,
    pitch_type_summ=pitch_type_l,
    **PARAMS,
    mask=True
)

In [28]:
j_beeks = suggest_pitches(
    target_pitcher_id=pid('Beeks, Jalen', pitcher_summ_l),
    pitcher_summ=pitcher_summ_l,
    pitch_type_summ=pitch_type_l,
    **PARAMS,
    mask=True
)

In [29]:
r_garcia = suggest_pitches(
    target_pitcher_id=pid('Garcia, Robert', pitcher_summ_l),
    pitcher_summ=pitcher_summ_l,        
    pitch_type_summ=pitch_type_l,
    **PARAMS
)

In [30]:
plot_pitch_clusters(j_beeks)

In [31]:
pitch_type_summ[['pfx_x', 'pfx_z']].describe()

,pfx_x,pfx_z
count,21713.000000,21713.000000
mean,-0.121726,0.468732
std,0.889955,0.692895
min,-1.985610,-1.728750
25%,-0.941282,0.068571
50%,-0.162688,0.509435
75%,0.610278,1.010765
max,2.380000,2.260000


In [32]:
pitch_type_summ[abs(pitch_type_summ['pfx_x']) > 2]

,pitch_type,pitcher,player_name,p_throws,game_year,release_speed,release_pos_x,release_pos_z,pfx_x,pfx_z,release_spin_rate,release_extension,release_pos_y,spin_axis,arm_angle,n
20989,ST,669302,"Gilbert, Logan",R,2026,75.8,-1.24,6.09,2.04,-0.10,2268.0,7.1,53.43,47.0,40.0,1
21118,ST,672456,"Montero, Keider",R,2025,83.7,-2.59,5.73,2.38,-0.23,2841.0,6.1,54.43,70.0,38.3,1


## Recommendation & Outlier Diagnostics

For each pitcher: how many pitch recommendations they get, and how many novel pitches
get trimmed as centroid outliers during clustering. Mirrors `suggest_pitches` but
computes the pairwise biomech distance matrix and the global pitch-characteristic
scaler **once** for the whole pool, then reuses them per pitcher (so it doesn't redo
the O(n^2) distance matrix on every call).

In [33]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from src.pitch_suggestions import _find_target, _collect_pitches, _tag_novelty, _cluster_novel


def diagnose_pool(
    pitcher_summ,
    pitch_type_summ,
    pitcher_ids=None,
    biomech_distance_threshold=PARAMS['biomech_distance_threshold'],
    novelty_distance_threshold=PARAMS['novelty_distance_threshold'],
    min_comp_usage_pct=PARAMS['min_comp_usage_pct'],
    min_pitches=PARAMS['min_pitches'],
    biomech_features=BIOMECH_FEATURES,
    pitch_features=PITCH_CHAR_FEATURES,
    **kwargs,  # forwarded to _cluster_novel (e.g. floor, mask)
):
    """One row per pitcher: #recommendations and #novel-pitch outliers trimmed.

    Mirrors the pipeline in suggest_pitches, but the expensive pieces -- the pairwise
    biomech distance matrix and the global pitch-characteristic scaler -- are computed
    ONCE for the whole pool and reused for every pitcher. n_outliers_removed is the
    number of novel pitches dropped by the centroid-outlier trimming inside
    _cluster_novel (n_novel_before_trim - n_novel_after_trim).

    n_recommendations counts final clusters (rows in the suggestions table), so a
    pitcher whose novel pitches collapse to a single cluster shows n_recommendations = 1.
    """
    # ---- computed once, shared across all pitchers ----
    biomech_dist = compute_euclidean_distances(
        pitcher_summ,
        features=biomech_features,
        label_cols=['pitcher', 'game_year'],
        min_pitches=min_pitches,
    )
    global_scaler = StandardScaler().fit(
        pitch_type_summ[pitch_features].dropna().values
    )

    if pitcher_ids is None:
        pitcher_ids = pitcher_summ['pitcher'].unique()

    rows = []
    for target_pitcher_id in pitcher_ids:
        target_row, target_year = _find_target(pitcher_summ, target_pitcher_id)
        rec = {
            'pitcher':             target_pitcher_id,
            'player_name':         None if target_row is None else target_row['player_name'],
            'status':              'pitcher_not_found',
            'n_recommendations':   0,
            'n_novel_before_trim': 0,
            'n_novel_after_trim':  0,
            'n_outliers_removed':  0,
        }
        if target_row is None:
            rows.append(rec)
            continue

        # ---- biomech comps: reuse the precomputed matrix (same logic as _find_biomech_comps) ----
        m = (
            ((biomech_dist['pitcher1'] == target_pitcher_id) & (biomech_dist['game_year1'] == target_year)) |
            ((biomech_dist['pitcher2'] == target_pitcher_id) & (biomech_dist['game_year2'] == target_year))
        )
        d = biomech_dist[m].copy()
        is_left = d['pitcher1'] == target_pitcher_id
        d['comp_pitcher'] = np.where(is_left, d['pitcher2'], d['pitcher1'])
        d['comp_year']    = np.where(is_left, d['game_year2'], d['game_year1'])
        target_dists = (
            d[['comp_pitcher', 'comp_year', 'distance']]
            .query('distance <= @biomech_distance_threshold and comp_pitcher != @target_pitcher_id')
            .sort_values('distance')
            .drop_duplicates(subset='comp_pitcher', keep='first')
            .reset_index(drop=True)
        )
        if target_dists.empty:
            rec['status'] = 'no_comps'
            rows.append(rec)
            continue

        target_pitches, comp_pitches = _collect_pitches(
            pitch_type_summ, target_pitcher_id, target_year, target_dists,
            pitch_features, min_comp_usage_pct, min_pitches,
        )
        if comp_pitches.empty:
            rec['status'] = 'no_comp_pitches'
            rows.append(rec)
            continue

        comp_pitches, novel = _tag_novelty(
            target_pitches, comp_pitches, pitch_features,
            novelty_distance_threshold, global_scaler,
        )
        rec['n_novel_before_trim'] = len(novel)
        if len(novel) < 4:
            rec['status'] = 'no_novel_pitches'
            rows.append(rec)
            continue

        novel = _cluster_novel(novel, global_scaler, pitch_features, **kwargs)
        rec['n_novel_after_trim'] = len(novel)
        rec['n_outliers_removed'] = rec['n_novel_before_trim'] - rec['n_novel_after_trim']
        if novel.empty:
            rec['status'] = 'no_novel_pitches'
            rows.append(rec)
            continue

        rec['status'] = 'ok'
        rec['n_recommendations'] = novel[['cluster_label', 'cluster']].drop_duplicates().shape[0]
        rows.append(rec)

    return pd.DataFrame(rows)

In [34]:
# Run the diagnostic across the whole left- and right-handed pools.
diag_l = diagnose_pool(pitcher_summ_l, pitch_type_l).assign(hand='L')
diag_r = diagnose_pool(pitcher_summ_r, pitch_type_r).assign(hand='R')
diag = pd.concat([diag_l, diag_r], ignore_index=True)

print(diag.shape[0], 'pitchers evaluated')
print('\nstatus breakdown:')
print(diag['status'].value_counts())
diag.head()

1440 pitchers evaluated

status breakdown:
status
ok                  1353
no_novel_pitches      72
no_comps              15
Name: count, dtype: int64


,pitcher,player_name,status,n_recommendations,n_novel_before_trim,n_novel_after_trim,n_outliers_removed,hand
0,431148,"Kazmir, Scott",ok,2,107,100,7,L
1,446321,"Detwiler, Ross",ok,2,27,24,3,L
2,448179,"Hill, Rich",ok,3,30,30,0,L
3,448281,"Doolittle, Sean",ok,2,76,72,4,L
4,452027,"Albers, Andrew",ok,3,37,35,2,L


In [35]:
# --- Only pitchers who actually got recommendations ---
ok = diag[diag['status'] == 'ok'].copy()

print('Distribution of # recommendations per pitcher:')
print(ok['n_recommendations'].value_counts().sort_index())

print('\nDistribution of # outliers removed per pitcher:')
print(ok['n_outliers_removed'].value_counts().sort_index())

# Pitchers who collapsed to a single recommendation (the 1-pitch case)
single = ok[ok['n_recommendations'] == 1]
print(f'\n{len(single)} pitcher(s) got exactly 1 recommendation')

# Full per-pitcher table, most-trimmed first
ok.sort_values(['n_outliers_removed', 'n_recommendations'], ascending=False)[
    ['pitcher', 'player_name', 'hand', 'n_recommendations',
     'n_novel_before_trim', 'n_novel_after_trim', 'n_outliers_removed']
].reset_index(drop=True)

Distribution of # recommendations per pitcher:
n_recommendations
1      1
2    867
3    333
4    152
Name: count, dtype: int64

Distribution of # outliers removed per pitcher:
n_outliers_removed
0      169
1      108
2       73
3       87
4       65
      ... 
176      1
181      1
186      1
208      1
210      1
Name: count, Length: 121, dtype: int64

1 pitcher(s) got exactly 1 recommendation


,pitcher,player_name,hand,n_recommendations,n_novel_before_trim,n_novel_after_trim,n_outliers_removed
0,664294,"Moreta, Dauri",R,2,511,301,210
1,656464,"Ginkel, Kevin",R,2,607,399,208
2,664918,"Keller, Kyle",R,2,776,590,186
3,644364,"Arano, Víctor",R,2,637,456,181
4,663432,"Rainey, Tanner",R,2,855,679,176
...,...,...,...,...,...,...,...
1348,691799,"Taylor, Grant",R,2,5,5,0
1349,695034,"Morris, Kade",R,2,43,43,0
1350,695243,"Miller, Mason",R,2,10,10,0
1351,695418,"Lord, Brad",R,2,4,4,0


In [36]:
ok[ok['n_recommendations'] == 4].head(20)

,pitcher,player_name,status,n_recommendations,n_novel_before_trim,n_novel_after_trim,n_outliers_removed,hand
5,452657,"Lester, Jon",ok,4,24,24,0,L
22,502154,"Britton, Zack",ok,4,153,147,6,L
23,502327,"Santiago, Héctor",ok,4,32,30,2,L
27,518617,"Diekman, Jake",ok,4,31,31,0,L
32,519242,"Sale, Chris",ok,4,9,9,0,L
35,542881,"Anderson, Tyler",ok,4,24,24,0,L
37,543045,"Conley, Adam",ok,4,42,41,1,L
49,554431,"Matzek, Tyler",ok,4,238,234,4,L
51,571510,"Boyd, Matthew",ok,4,54,54,0,L
57,571927,"Matz, Steven",ok,4,79,78,1,L
